# Flutter Uygulamalarını Uluslararasılaştırma

**Neler öğreneceksiniz?** \* Cihazın yerel ayarını (kullanıcının tercih
ettiği dil) nasıl takip edersiniz. \* Yerel ayara özgü Material veya
Cupertino widget’larını nasıl etkinleştirirsiniz. \* Yerel ayara özgü
uygulama değerlerini nasıl yönetirsiniz. \* Bir uygulamanın desteklediği
yerel ayarları (locales) nasıl tanımlarsınız.

Uygulamanız başka bir dili konuşan kullanıcılara sunulacaksa, onu
uluslararası hale getirmeniz (internationalize) gerekir. Bu, uygulamayı,
uygulamanın desteklediği her dil veya yerel ayar için metin ve düzen
(layout) gibi değerleri yerelleştirmeyi (localize) mümkün kılacak
şekilde yazmanız gerektiği anlamına gelir. Flutter,
uluslararasılaştırmaya yardımcı olan widget’lar ve sınıflar sağlar ve
Flutter kütüphanelerinin kendileri de uluslararasılaştırılmıştır.

Bu sayfa, çoğu uygulama bu şekilde yazıldığı için `MaterialApp` ve
`CupertinoApp` sınıflarını kullanarak bir Flutter uygulamasını
yerelleştirmek için gerekli kavramları ve iş akışlarını kapsar. Ancak,
daha düşük seviyeli `WidgetsApp` sınıfı kullanılarak yazılan uygulamalar
da aynı sınıflar ve mantık kullanılarak uluslararasılaştırılabilir.

## Flutter’da yerelleştirmeye giriş

Bu bölüm, yeni bir Flutter uygulamasının nasıl oluşturulacağına ve
uluslararasılaştırılacağına dair bir eğitimin yanı sıra hedef platformun
gerektirebileceği ek kurulumları sağlar.

Bu örneğin kaynak kodunu
[gen_l10n_example](https://github.com/flutter/website/tree/main/examples/internationalization/gen_l10n_example)
içinde bulabilirsiniz.

### Uluslararasılaştırılmış bir uygulama kurma: Flutter_localizations paketi

Varsayılan olarak, Flutter yalnızca ABD İngilizcesi yerelleştirmelerini
sağlar. Diğer diller için destek eklemek üzere, bir uygulamanın ek
`MaterialApp` (veya `CupertinoApp`) özellikleri belirtmesi ve
`flutter_localizations` adlı bir paketi dahil etmesi gerekir.

Başlamak için, `flutter create` komutuyla seçtiğiniz bir dizinde yeni
bir Flutter uygulaması oluşturun.

``` bash
flutter create <flutter_uygulamasının_adı>
```

`flutter_localizations` kullanmak için, paketi ve ayrıca `intl` paketini
`pubspec.yaml` dosyanıza bağımlılık olarak ekleyin:

``` bash
flutter pub add flutter_localizations --sdk=flutter
flutter pub add intl:any
```

Bu, aşağıdaki girdilere sahip bir `pubspec.yaml` dosyası oluşturur:

``` yaml
dependencies:
  flutter:
    sdk: flutter
  flutter_localizations:
    sdk: flutter
  intl: any
```

Ardından `flutter_localizations` kütüphanesini içe aktarın ve
`MaterialApp` veya `CupertinoApp` için `localizationsDelegates` ve
`supportedLocales` özelliklerini belirtin:

``` dart
import 'package:flutter_localizations/flutter_localizations.dart';

return const MaterialApp(
  title: 'Localizations Sample App',
  localizationsDelegates: [
    GlobalMaterialLocalizations.delegate,
    GlobalWidgetsLocalizations.delegate,
    GlobalCupertinoLocalizations.delegate,
  ],
  supportedLocales: [
    Locale('en'), // İngilizce
    Locale('es'), // İspanyolca
  ],
  home: MyHomePage(),
);
```

`flutter_localizations` paketini tanıttıktan ve önceki kodu ekledikten
sonra, `Material` ve `Cupertino` paketleri artık desteklenen yerel
ayarlardan birinde doğru şekilde yerelleştirilmiş olmalıdır. Widget’lar,
doğru soldan sağa veya sağdan sola düzen ile birlikte yerelleştirilmiş
mesajlara uyarlanmalıdır.

Hedef platformun yerel ayarını İspanyolca’ya (es) değiştirmeyi deneyin;
mesajlar yerelleştirilmiş olmalıdır.

`WidgetsApp` tabanlı uygulamalar, `GlobalMaterialLocalizations.delegate`
gerekli olmaması dışında benzerdir.

Tam `Locale.fromSubtags` yapıcısı, `scriptCode` desteği sağladığı için
tercih edilir, ancak `Locale` varsayılan yapıcısı da hala tamamen
geçerlidir.

`localizationsDelegates` listesinin öğeleri, yerelleştirilmiş değer
koleksiyonları üreten fabrikalardır.
`GlobalMaterialLocalizations.delegate`, Material Components kütüphanesi
için yerelleştirilmiş dizeler ve diğer değerleri sağlar.
`GlobalWidgetsLocalizations.delegate`, widget kütüphanesi için
varsayılan metin yönünü (soldan sağa veya sağdan sola) tanımlar.

Bu uygulama özellikleri, bağlı oldukları türler ve
uluslararasılaştırılmış Flutter uygulamalarının tipik olarak nasıl
yapılandırıldığı hakkında daha fazla bilgi bu sayfada ele alınmaktadır.

### Yerel ayarı (Locale) geçersiz kılma

`Localizations.override`, uygulamanızın bir bölümünün cihazınız için
yapılandırılan yerel ayardan farklı bir yerel ayara yerelleştirilmesi
gerektiği (genellikle nadir) durumlar için izin veren `Localizations`
widget’ı için bir fabrika yapıcısıdır.

Bu davranışı gözlemlemek için `Localizations.override` çağrısı ve basit
bir `CalendarDatePicker` ekleyin:

``` dart
Widget build(BuildContext context) {
  return Scaffold(
    appBar: AppBar(title: Text(widget.title)),
    body: Center(
      child: Column(
        mainAxisAlignment: MainAxisAlignment.center,
        children: <Widget>[
          // Aşağıdaki kodu ekleyin
          Localizations.override(
            context: context,
            locale: const Locale('es'),
            // Doğru BuildContext'i almak için bir Builder kullanıyoruz.
            // Alternatif olarak, yeni bir widget oluşturabilirsiniz ve
            // Localizations.override güncellenmiş BuildContext'i yeni widget'a iletecektir.
            child: Builder(
              builder: (context) {
                // Uluslararasılaştırılmış bir Material widget'ı için basit bir örnek.
                return CalendarDatePicker(
                  initialDate: DateTime.now(),
                  firstDate: DateTime(1900),
                  lastDate: DateTime(2100),
                  onDateChanged: (value) {},
                );
              },
            ),
          ),
        ],
      ),
    ),
  );
}
```

Uygulamayı “Hot reload” yapın; `CalendarDatePicker` widget’ı İspanyolca
olarak yeniden oluşturulmalıdır.

## Kendi yerelleştirilmiş mesajlarınızı ekleme

`flutter_localizations` paketini ekledikten sonra yerelleştirmeyi
yapılandırabilirsiniz. Uygulamanıza yerelleştirilmiş metin eklemek için
aşağıdaki talimatları tamamlayın:

1.  `flutter_localizations` tarafından sabitlenen sürümü çekerek `intl`
    paketini bir bağımlılık olarak ekleyin:

``` bash
flutter pub add intl:any
```

1.  `pubspec.yaml` dosyasını açın ve `generate` bayrağını etkinleştirin.
    Bu bayrak, pubspec dosyasındaki `flutter` bölümünde bulunur.

``` yaml
# Aşağıdaki bölüm Flutter'a özgüdür.
flutter:
  generate: true # Bu satırı ekleyin
```

1.  Flutter projesinin kök dizinine yeni bir yaml dosyası ekleyin. Bu
    dosyayı `l10n.yaml` olarak adlandırın ve aşağıdaki içeriği ekleyin:

``` yaml
arb-dir: lib/l10n
template-arb-file: app_en.arb
output-localization-file: app_localizations.dart
```

Bu dosya yerelleştirme aracını yapılandırır. Bu örnekte şunları
yaptınız: \* Uygulama Kaynak Paketi ([App Resource
Bundle](https://github.com/google/app-resource-bundle) - .arb) girdi
dosyalarını `${FLUTTER_PROJECT}/lib/l10n` dizinine koydunuz. `.arb`
dosyaları uygulamanız için yerelleştirme kaynakları sağlar. \* İngilizce
şablonunu `app_en.arb` olarak ayarladınız. \* Flutter’a
yerelleştirmeleri `app_localizations.dart` dosyasında oluşturmasını
söylediniz.

1.  `${FLUTTER_PROJECT}/lib/l10n` içine `app_en.arb` şablon dosyasını
    ekleyin. Örneğin:

``` json
{
  "helloWorld": "Hello World!",
  "@helloWorld": {
    "description": "The conventional newborn programmer greeting"
  }
}
```

1.  Aynı dizine `app_es.arb` adlı başka bir paket dosyası ekleyin. Bu
    dosyaya, aynı mesajın İspanyolca çevirisini ekleyin.

``` json
{
    "helloWorld": "¡Hola Mundo!"
}
```

1.  Şimdi, `flutter pub get` veya `flutter run` komutunu çalıştırın; kod
    oluşturma (codegen) işlemi otomatik olarak gerçekleşir. Oluşturulan
    dosyaları `arb-dir` veya `output-dir` seçenekleriyle belirttiğiniz
    yoldaki dizinde bulmalısınız. Alternatif olarak, uygulamayı
    çalıştırmadan aynı dosyaları oluşturmak için `flutter gen-l10n`
    komutunu da çalıştırabilirsiniz.
2.  `MaterialApp` yapıcısı çağrınıza `app_localizations.dart` ve
    `AppLocalizations.delegate` için içe aktarma ifadesini ekleyin:

``` dart
import 'l10n/app_localizations.dart';

return const MaterialApp(
  title: 'Localizations Sample App',
  localizationsDelegates: [
    AppLocalizations.delegate, // Bu satırı ekleyin
    GlobalMaterialLocalizations.delegate,
    GlobalWidgetsLocalizations.delegate,
    GlobalCupertinoLocalizations.delegate,
  ],
  supportedLocales: [
    Locale('en'), // İngilizce
    Locale('es'), // İspanyolca
  ],
  home: MyHomePage(),
);
```

`AppLocalizations` sınıfı ayrıca otomatik olarak oluşturulan
`localizationsDelegates` ve `supportedLocales` listelerini de sağlar.
Bunları manuel olarak sağlamak yerine kullanabilirsiniz:

``` dart
const MaterialApp(
  title: 'Localizations Sample App',
  localizationsDelegates: AppLocalizations.localizationsDelegates,
  supportedLocales: AppLocalizations.supportedLocales,
);
```

1.  Material uygulaması başladıktan sonra, `AppLocalizations` sınıfını
    uygulamanızın herhangi bir yerinde kullanabilirsiniz:

``` dart
appBar: AppBar(
  // [AppBar] başlık metni, hedef platformun sistem yerel ayarlarına
  // göre mesajını güncellemelidir.
  // İngilizce ve İspanyolca yerel ayarlar arasında geçiş yapmak
  // bu metnin güncellenmesine neden olmalıdır.
  title: Text(AppLocalizations.of(context)!.helloWorld),
),
```

> **Not:** `AppLocalizations` sınıfını başlatmak için Material
> uygulamasının gerçekten başlatılmış olması gerekir. Uygulama henüz
> başlamadıysa, `AppLocalizations.of(context)!.helloWorld` bir null
> istisnasına (exception) neden olur.

Bu kod, hedef cihazın yerel ayarı İngilizce olarak ayarlanmışsa “Hello
World!”, İspanyolca olarak ayarlanmışsa “¡Hola Mundo!” görüntüleyen bir
`Text` widget’ı oluşturur. `.arb` dosyalarında, her girişin anahtarı
alıcının (getter) yöntem adı olarak kullanılırken, o girişin değeri
yerelleştirilmiş mesajı içerir.

`gen_l10n_example` bu aracı kullanır.

Cihaz uygulama açıklamanızı yerelleştirmek için, yerelleştirilmiş dizeyi
`MaterialApp.onGenerateTitle`’a iletin:

``` dart
return MaterialApp(
  onGenerateTitle: (context) => DemoLocalizations.of(context).title,
);
```

### Yer tutucular, çoğullar ve seçimler

> **İpucu:** VS Code kullanırken `arb-editor` uzantısını ekleyin. Bu
> uzantı, `.arb` şablon dosyalarını düzenlemeye yardımcı olmak için
> sözdizimi vurgulama, snippet’ler, teşhisler ve hızlı düzeltmeler
> ekler.

Ayrıca, bir alıcı (getter) yerine bir yöntem oluşturmak için bir
`yer tutucu` (placeholder) kullanan özel bir sözdizimi ile uygulama
değerlerini bir mesaja dahil edebilirsiniz. Geçerli bir Dart tanımlayıcı
adı olması gereken bir yer tutucu, `AppLocalizations` kodundaki
oluşturulan yöntemde konumsal bir parametre haline gelir. Bir yer tutucu
adını aşağıdaki gibi süslü parantezler içine alarak tanımlayın:

`"{placeholderName}"`

Her yer tutucuyu uygulamanın `.arb` dosyasındaki `placeholders`
nesnesinde tanımlayın. Örneğin, bir `userName` parametresine sahip bir
selamlama mesajı tanımlamak için `lib/l10n/app_en.arb` dosyasına
aşağıdakini ekleyin:

``` json
"hello": "Hello {userName}",
"@hello": {
  "description": "A message with a single parameter",
  "placeholders": {
    "userName": {
      "type": "String",
      "example": "Bob"
    }
  }
}
```

Bu kod parçacığı, `AppLocalizations.of(context)` nesnesine bir `hello`
yöntemi çağrısı ekler ve yöntem `String` türünde bir parametre kabul
eder; `hello` yöntemi bir dize döndürür. `AppLocalizations` dosyasını
yeniden oluşturun.

`Builder` içine geçirilen kodu aşağıdakiyle değiştirin:

``` dart
// Examples of internationalized strings.
return Column(
  children: <Widget>[
    // Returns 'Hello John'
    Text(AppLocalizations.of(context)!.hello('John')),
  ],
);
```

Birden çok değer belirtmek için sayısal yer tutucular da
kullanabilirsiniz. Farklı dillerin kelimeleri çoğullaştırmak (pluralize)
için farklı yolları vardır. Sözdizimi ayrıca bir kelimenin *nasıl*
çoğullaştırılacağını belirtmeyi de destekler. `Çoğullaştırılmış` bir
mesaj, kelimenin farklı durumlarda nasıl çoğullaştırılacağını gösteren
bir `num` parametresi içermelidir. Örneğin İngilizce, “person”
kelimesini “people” olarak çoğullaştırır, ancak bu yeterli değildir.
`message0` çoğulu “no people” veya “zero people” olabilir. `messageFew`
çoğulu “several people”, “some people” veya “a few people” olabilir.
`messageMany` çoğulu “most people”, “many people” veya “a crowd”
olabilir. Yalnızca daha genel olan `messageOther` alanı zorunludur.
Aşağıdaki örnek hangi seçeneklerin mevcut olduğunu gösterir:

`"{countPlaceholder, plural, =0{message0} =1{message1} =2{message2} few{messageFew} many{messageMany} other{messageOther}}"`

Önceki ifade, `countPlaceholder` değerine karşılık gelen mesaj
varyasyonu (message0, message1, …) ile değiştirilir. Yalnızca
`messageOther` alanı zorunludur.

Aşağıdaki örnek, “wombat” kelimesini çoğullaştıran bir mesaj tanımlar:

``` json
"nWombats": "{count, plural, =0{no wombats} =1{1 wombat} other{{count} wombats}}",
"@nWombats": {
  "description": "A plural message",
  "placeholders": {
    "count": {
      "type": "num",
      "format": "compact"
    }
  }
}
```

`count` parametresini geçirerek bir çoğul yöntem kullanın:

``` dart
// Examples of internationalized strings.
return Column(
  children: <Widget>[
    ...
    // Returns 'no wombats'
    Text(AppLocalizations.of(context)!.nWombats(0)),
    // Returns '1 wombat'
    Text(AppLocalizations.of(context)!.nWombats(1)),
    // Returns '5 wombats'
    Text(AppLocalizations.of(context)!.nWombats(5)),
  ],
);
```

Çoğullara benzer şekilde, bir `String` yer tutucuya dayalı olarak da bir
değer seçebilirsiniz. Bu genellikle cinsiyetli dilleri desteklemek için
kullanılır. Sözdizimi aşağıdaki gibidir:

`"{selectPlaceholder, select, case{message} ... other{messageOther}}"`

Sonraki örnek, cinsiyete göre bir zamir seçen bir mesaj tanımlar:

``` json
"pronoun": "{gender, select, male{he} female{she} other{they}}",
"@pronoun": {
  "description": "A gendered message",
  "placeholders": {
    "gender": {
      "type": "String"
    }
  }
}
```

Cinsiyet dizesini bir parametre olarak geçirerek bu özelliği kullanın:

``` dart
// Examples of internationalized strings.
return Column(
  children: <Widget>[
    ...
    // Returns 'he'
    Text(AppLocalizations.of(context)!.pronoun('male')),
    // Returns 'she'
    Text(AppLocalizations.of(context)!.pronoun('female')),
    // Returns 'they'
    Text(AppLocalizations.of(context)!.pronoun('other')),
  ],
);
```

`select` ifadelerini kullanırken, parametre ile gerçek değer arasındaki
karşılaştırmanın büyük/küçük harf duyarlı olduğunu unutmayın. Yani,
`AppLocalizations.of(context)!.pronoun("Male")` varsayılan olarak
“other” durumuna düşer ve “they” döndürür.

### Kaçış (Escaping) sözdizimi

Bazen `{` ve `}` gibi belirteçleri normal karakterler olarak kullanmanız
gerekebilir. Bu tür belirteçlerin ayrıştırılmasını (parsing) önlemek
için, `l10n.yaml` dosyasına aşağıdakini ekleyerek `use-escaping`
bayrağını etkinleştirin:

``` yaml
use-escaping: true
```

Ayrıştırıcı (parser), bir çift tek tırnak içine alınmış herhangi bir
karakter dizisini görmezden gelir. Normal bir tek tırnak karakteri
kullanmak için, bir çift ardışık tek tırnak kullanın. Örneğin, aşağıdaki
metin bir Dart `String`’e dönüştürülür:

``` json
{
  "helloWorld": "Hello! '{Isn''t}' this a wonderful day?"
}
```

Ortaya çıkan dize şu şekildedir:

`"Hello! {Isn't} this a wonderful day?"`

### Sayılar ve para birimleri içeren mesajlar

Para birimi değerlerini temsil edenler de dahil olmak üzere sayılar,
farklı yerel ayarlarda çok farklı görüntülenir. `flutter_localizations`
içindeki yerelleştirme oluşturma aracı, sayıları yerel ayara ve istenen
biçime göre biçimlendirmek için `intl` paketindeki `NumberFormat`
sınıfını kullanır.

`int`, `double` ve `num` türleri aşağıdaki `NumberFormat` yapıcılarından
herhangi birini kullanabilir:

| Mesaj “format” değeri   | 1200000 için çıktı |
|-------------------------|--------------------|
| compact                 | “1.2M”             |
| compactCurrency\*       | “\$1.2M”           |
| compactSimpleCurrency\* | “\$1.2M”           |
| compactLong             | “1.2 million”      |
| currency\*              | “USD1,200,000.00”  |
| decimalPattern          | “1,200,000”        |
| decimalPatternDigits\*  | “1,200,000”        |
| decimalPercentPattern\* | “120,000,000%”     |
| percentPattern          | “120,000,000%”     |
| scientificPattern       | “1E6”              |
| simpleCurrency\*        | “\$1,200,000”      |

Tablodaki yıldızlı `NumberFormat` yapıcıları isteğe bağlı, adlandırılmış
parametreler sunar. Bu parametreler, yer tutucunun `optionalParameters`
nesnesinin değeri olarak belirtilebilir. Örneğin, `compactCurrency` için
isteğe bağlı `decimalDigits` parametresini belirtmek üzere
`lib/l10n/app_en.arb` dosyasında aşağıdaki değişiklikleri yapın:

``` json
"numberOfDataPoints": "Number of data points: {value}",
"@numberOfDataPoints": {
  "description": "A message with a formatted int parameter",
  "placeholders": {
    "value": {
      "type": "int",
      "format": "compactCurrency",
      "optionalParameters": {
        "decimalDigits": 2
      }
    }
  }
}
```

### Tarih içeren mesajlar

Tarih dizeleri, hem yerel ayara hem de uygulamanın ihtiyaçlarına bağlı
olarak birçok farklı şekilde biçimlendirilir.

`DateTime` türündeki yer tutucu değerleri, `intl` paketindeki
`DateFormat` ile biçimlendirilir.

`DateFormat` fabrika yapıcılarının adlarıyla tanımlanan 41 format
varyasyonu vardır. Aşağıdaki örnekte, `helloWorldOn` mesajında görünen
`DateTime` değeri `DateFormat.yMd` ile biçimlendirilmiştir:

``` json
"helloWorldOn": "Hello World on {date}",
"@helloWorldOn": {
  "description": "A message with a date parameter",
  "placeholders": {
    "date": {
      "type": "DateTime",
      "format": "yMd"
    }
  }
}
```

Yerel ayarın ABD İngilizcesi olduğu bir uygulamada, aşağıdaki ifade
“7/9/1959” sonucunu üretir. Rusça bir yerel ayarda ise “9.07.1959”
sonucunu üretir.

``` dart
AppLocalizations.of(context).helloWorldOn(DateTime.utc(1959, 7, 9))
```

## iOS için Yerelleştirme: iOS uygulama paketini güncelleme

Yerelleştirmeler Flutter tarafından yönetilse de, desteklenen dilleri
Xcode projesine eklemeniz gerekir. Bu, App Store girişinizin desteklenen
dilleri doğru şekilde görüntülemesini sağlar.

Uygulamanızın desteklediği yerel ayarları yapılandırmak için aşağıdaki
talimatları kullanın:

1.  Projenizin `ios/Runner.xcodeproj` Xcode dosyasını açın.
2.  **Project Navigator** (Proje Gezgini) içinde, **Projects**
    (Projeler) altındaki `Runner` proje dosyasını seçin.
3.  Proje düzenleyicisinde **Info** sekmesini seçin.
4.  **Localizations** (Yerelleştirmeler) bölümünde, projenize
    desteklenen dilleri ve bölgeleri eklemek için **Ekle** düğmesine (+)
    tıklayın. Dosyaları ve referans dilini seçmeniz istendiğinde, sadece
    **Finish** (Bitir) seçeneğini seçin.

Xcode otomatik olarak boş `.strings` dosyaları oluşturur ve
`ios/Runner.xcodeproj/project.pbxproj` dosyasını günceller. Bu dosyalar,
uygulamanızın hangi dilleri ve bölgeleri desteklediğini belirlemek için
App Store tarafından kullanılır.

## Daha fazla özelleştirme için ileri düzey konular

Bu bölüm, yerelleştirilmiş bir Flutter uygulamasını özelleştirmenin ek
yollarını kapsar.

### İleri düzey yerel ayar (Locale) tanımı

Birden çok varyantı olan bazı diller, doğru şekilde ayırt etmek için
sadece bir dil kodundan fazlasını gerektirir.

Örneğin, Çincenin tüm varyantlarını tam olarak ayırt etmek; dil kodunu,
yazı sistemi kodunu (script code) ve ülke kodunu belirtmeyi gerektirir.
Bunun nedeni, basitleştirilmiş ve geleneksel yazı sisteminin varlığı ve
aynı yazı sistemi türünde karakterlerin yazılış şeklindeki bölgesel
farklılıklardır.

`CN`, `TW` ve `HK` ülke kodları için Çincenin her varyantını tam olarak
ifade etmek amacıyla, desteklenen yerel ayarlar listesi şunları
içermelidir:

``` dart
supportedLocales: [
  Locale.fromSubtags(languageCode: 'zh'), // genel Çince 'zh'
  Locale.fromSubtags(
    languageCode: 'zh',
    scriptCode: 'Hans',
  ), // genel basitleştirilmiş Çince 'zh_Hans'
  Locale.fromSubtags(
    languageCode: 'zh',
    scriptCode: 'Hant',
  ), // genel geleneksel Çince 'zh_Hant'
  Locale.fromSubtags(
    languageCode: 'zh',
    scriptCode: 'Hans',
    countryCode: 'CN',
  ), // 'zh_Hans_CN'
  Locale.fromSubtags(
    languageCode: 'zh',
    scriptCode: 'Hant',
    countryCode: 'TW',
  ), // 'zh_Hant_TW'
  Locale.fromSubtags(
    languageCode: 'zh',
    scriptCode: 'Hant',
    countryCode: 'HK',
  ), // 'zh_Hant_HK'
],
```

Bu açık ve tam tanım, uygulamanızın bu ülke kodlarının tüm
kombinasyonlarına tam nüanslı yerelleştirilmiş içeriği ayırt
edebilmesini ve sunabilmesini sağlar. Kullanıcının tercih ettiği yerel
ayar belirtilmemişse, Flutter en yakın eşleşmeyi seçer; bu da muhtemelen
kullanıcının beklediğinden farklılıklar içerir. Flutter yalnızca
`supportedLocales` içinde tanımlanan yerel ayarlara çözümleme yapar ve
yaygın olarak kullanılan diller için `scriptCode` ile ayırt edilen
yerelleştirilmiş içerik sağlar. Desteklenen yerel ayarların ve tercih
edilen yerel ayarların nasıl çözümlendiği hakkında bilgi için
`Localizations` bölümüne bakın.

Çince birincil bir örnek olsa da, Fransızca (`fr_FR`, `fr_CA`) gibi
diğer diller de daha nüanslı yerelleştirme için tam olarak ayırt
edilmelidir.

### Yerel ayarı takip etme: Locale sınıfı ve Localizations widget’ı

`Locale` sınıfı kullanıcının dilini tanımlar. Mobil cihazlar, genellikle
bir sistem ayarları menüsü kullanarak tüm uygulamalar için yerel ayarı
ayarlamayı destekler. Uluslararasılaştırılmış uygulamalar, yerel ayara
özgü değerleri görüntüleyerek yanıt verir. Örneğin, kullanıcı cihazın
yerel ayarını İngilizceden Fransızcaya değiştirirse, orijinal olarak
“Hello World” görüntüleyen bir `Text` widget’ı “Bonjour le monde” ile
yeniden oluşturulur.

`Localizations` widget’ı, alt öğesi (child) için yerel ayarı ve alt
öğenin bağlı olduğu yerelleştirilmiş kaynakları tanımlar. `WidgetsApp`
widget’ı bir `Localizations` widget’ı oluşturur ve sistemin yerel ayarı
değişirse onu yeniden oluşturur.

Uygulamanın mevcut yerel ayarını her zaman `Localizations.localeOf()`
ile arayabilirsiniz:

``` dart
Locale myLocale = Localizations.localeOf(context);
```

### Uygulamanın supportedLocales parametresini belirtme

`flutter_localizations` kütüphanesi birçok dili ve dil varyantını
desteklese de, varsayılan olarak yalnızca İngilizce dil çevirileri
mevcuttur. Tam olarak hangi dillerin destekleneceğine karar vermek
geliştiriciye bağlıdır.

`MaterialApp` `supportedLocales` parametresi yerel ayar değişikliklerini
sınırlar. Kullanıcı cihazındaki yerel ayar ayarını değiştirdiğinde,
uygulamanın `Localizations` widget’ı yalnızca yeni yerel ayar bu
listenin bir üyesiyse buna uyar. Cihaz yerel ayarı için tam bir eşleşme
bulunamazsa, eşleşen bir `languageCode`’a sahip ilk desteklenen yerel
ayar kullanılır. Bu da başarısız olursa, `supportedLocales` listesinin
ilk öğesi kullanılır.

Farklı bir “yerel ayar çözümleme” yöntemi kullanmak isteyen bir
uygulama, bir `localeResolutionCallback` sağlayabilir. Örneğin,
uygulamanızın kullanıcının seçtiği yerel ayarı koşulsuz olarak kabul
etmesini sağlamak için:

``` dart
MaterialApp(
  localeResolutionCallback: (locale, supportedLocales) {
    return locale;
  },
);
```

# l10n.yaml Dosyasını Yapılandırma

`l10n.yaml` dosyası, `gen-l10n` aracını yapılandırarak şunları
belirtmenize olanak tanır: \* Tüm girdi dosyalarının nerede bulunduğu \*
Tüm çıktı dosyalarının nerede oluşturulması gerektiği \* Yerelleştirme
temsilcilerinize (delegates) hangi Dart sınıf adının verileceği

Seçeneklerin tam listesi için komut satırında `flutter gen-l10n --help`
komutunu çalıştırın veya aşağıdaki tabloya bakın:

| Seçenek | Açıklama |
|:-----------------------------------|:-----------------------------------|
| `arb-dir` | Şablon ve çevrilmiş arb dosyalarının bulunduğu dizin. Varsayılan `lib/l10n` dizinidir. |
| `output-dir` | Oluşturulan yerelleştirme sınıflarının yazıldığı dizin. Bu seçenek yalnızca yerelleştirme kodunu Flutter projesinde başka bir yerde oluşturmak istiyorsanız geçerlidir. Ayrıca `synthetic-package` bayrağını `false` olarak ayarlamanız gerekir.<br><br>Uygulama, `output-localization-file` seçeneğinde belirtilen dosyayı bu dizinden içe aktarmalıdır (import). Belirtilmezse, bu varsayılan olarak `arb-dir` içinde belirtilen girdi diziniyle aynı dizine ayarlanır. |
| `template-arb-file` | Dart yerelleştirme ve mesaj dosyalarını oluşturmak için temel olarak kullanılan şablon arb dosyası. Varsayılan `app_en.arb` dosyasıdır. |
| `output-localization-file` | Çıktı yerelleştirme ve yerelleştirme temsilcisi (delegate) sınıfları için dosya adı. Varsayılan `app_localizations.dart` dosyasıdır. |
| `untranslated-messages-file` | Henüz çevrilmemiş yerelleştirme mesajlarını tanımlayan bir dosyanın konumu. Bu seçeneği kullanmak, hedef konumda aşağıdaki formatta bir JSON dosyası oluşturur:<br><br>`"locale": ["message_1", "message_2" ... "message_n"]`<br><br>Bu seçenek belirtilmezse, çevrilmemiş mesajların bir özeti komut satırında yazdırılır. |
| `output-class` | Çıktı yerelleştirme ve yerelleştirme temsilcisi sınıfları için kullanılacak Dart sınıf adı. Varsayılan `AppLocalizations` sınıfıdır. |
| `preferred-supported-locales` | Uygulama için tercih edilen desteklenen yerel ayarların listesi. Varsayılan olarak araç, desteklenen yerel ayarlar listesini alfabetik sıraya göre oluşturur. Farklı bir yerel ayarı varsayılan yapmak için bu bayrağı kullanın.<br><br>Örneğin, bir cihaz destekliyorsa Amerikan İngilizcesini varsayılan yapmak için `[ en_US ]` değerini geçirin. |
| `header` | Oluşturulan Dart yerelleştirme dosyalarının başına eklenecek başlık. Bu seçenek bir dize (string) alır.<br><br>Örneğin, oluşturulan Dart dosyasının başına bu dizeyi eklemek için `"/// All localized files."` ifadesini geçirin.<br><br>Alternatif olarak, daha uzun başlıklar için bir metin dosyası geçirmek üzere `header-file` seçeneğine göz atın. |
| `header-file` | Oluşturulan Dart yerelleştirme dosyalarının başına eklenecek başlık. Bu seçeneğin değeri, oluşturulan her Dart dosyasının en üstüne eklenecek başlık metnini içeren dosyanın adıdır.<br><br>Alternatif olarak, daha basit bir başlık için bir dize geçirmek üzere `header` seçeneğine göz atın.<br><br>Bu dosya `arb-dir` içinde belirtilen dizine yerleştirilmelidir. |
| `[no-]use-deferred-loading` | Yerel ayarların ertelenmiş (deferred) olarak içe aktarıldığı Dart yerelleştirme dosyasının oluşturulup oluşturulmayacağını belirtir; bu, Flutter web’de her yerel ayarın tembel yüklenmesine (lazy loading) izin verir.<br><br>Bu, JavaScript paketinin boyutunu azaltarak bir web uygulamasının ilk başlangıç süresini kısaltabilir. Bu bayrak `true` olarak ayarlandığında, belirli bir yerel ayara ait mesajlar yalnızca Flutter uygulaması tarafından ihtiyaç duyulduğunda indirilir ve yüklenir. Çok sayıda farklı yerel ayara ve birçok yerelleştirme dizesine sahip projeler için yüklemeyi ertelemek performansı artırabilir. Az sayıda yerel ayara sahip projeler için fark ihmal edilebilir düzeydedir ve yerelleştirmeleri uygulamanın geri kalanıyla birlikte paketlemeye kıyasla başlangıcı yavaşlatabilir.<br><br>Bu bayrağın mobil veya masaüstü gibi diğer platformları etkilemediğini unutmayın. |
| `gen-inputs-and-outputs-list` | Belirtildiğinde araç, aracın girdilerini ve çıktılarını içeren `gen_l10n_inputs_and_outputs.json` adında bir JSON dosyası oluşturur.<br><br>Bu, en son yerelleştirme seti oluşturulurken Flutter projesinin hangi dosyalarının kullanıldığını takip etmek için yararlı olabilir. Örneğin, Flutter aracının derleme sistemi, çalışırken yeniden yükleme (hot reload) sırasında `gen_l10n` aracının ne zaman çağrılacağını takip etmek için bu dosyayı kullanır.<br><br>Bu seçeneğin değeri, JSON dosyasının oluşturulduğu dizindir. Null olduğunda JSON dosyası oluşturulmaz. |
| `synthetic-package` | Oluşturulan çıktı dosyalarının sentetik bir paket olarak mı yoksa Flutter projesindeki belirtilen bir dizinde mi oluşturulacağını belirler. Bu bayrak varsayılan olarak `true` değerindedir. `synthetic-package` `false` olarak ayarlandığında, yerelleştirme dosyalarını varsayılan olarak `arb-dir` tarafından belirtilen dizinde oluşturur. Eğer `output-dir` belirtilirse, dosyalar orada oluşturulur. |
| `project-dir` | Belirtildiğinde araç, bu seçeneğe geçirilen yolu kök Flutter projesinin dizini olarak kullanır.<br><br>Null olduğunda, mevcut çalışma dizinine giden göreli yol kullanılır. |
| `[no-]required-resource-attributes` | Tüm kaynak kimliklerinin (resource ids) karşılık gelen bir kaynak özniteliği (attribute) içermesini gerektirir.<br><br>Varsayılan olarak basit mesajlar meta veri gerektirmez, ancak mesajın anlamı hakkında okuyuculara bağlam sağladığı için bu şiddetle tavsiye edilir.<br><br>Çoğul (plural) mesajlar için kaynak öznitelikleri hala zorunludur. |
| `[no-]nullable-getter` | Yerelleştirme sınıfı alıcısının (getter) null olabilir (nullable) olup olmadığını belirtir.<br><br>Varsayılan olarak, geriye dönük uyumluluk için `Localizations.of(context)` null bir değer döndürecek şekilde bu değer `true`’dur. Bu değer `false` ise, `Localizations.of(context)` dönüş değerinde bir null kontrolü yapılır ve kullanıcı kodunda null kontrolü yapma ihtiyacı ortadan kalkar. |
| `[no-]format` | Belirtildiğinde, yerelleştirme dosyaları oluşturulduktan sonra `dart format` komutu çalıştırılır. |
| `use-escaping` | Kaçış sözdizimi olarak tek tırnak kullanımının etkinleştirilip etkinleştirilmeyeceğini belirtir. |
| `[no-]suppress-warnings` | Belirtildiğinde, tüm uyarılar bastırılır. |
| `[no-]relax-syntax` | Belirtildiğinde, sözdizimi gevşetilir; böylece özel karakter “{”, geçerli bir yer tutucu (placeholder) tarafından takip edilmiyorsa bir dize olarak ele alınır ve “}”, özel bir karakter olarak ele alınan önceki bir “{” karakterini kapatmıyorsa bir dize olarak ele alınır. |
| `[no-]use-named-parameters` | Oluşturulan yerelleştirme yöntemleri için adlandırılmış parametrelerin kullanılıp kullanılmayacağı. |

## Flutter’da uluslararasılaştırma nasıl çalışır?

Bu bölüm, Flutter’da yerelleştirmelerin nasıl çalıştığına dair teknik
ayrıntıları kapsar. Kendi yerelleştirilmiş mesaj setinizi desteklemeyi
planlıyorsanız, aşağıdaki içerik yararlı olacaktır. Aksi takdirde, bu
bölümü atlayabilirsiniz.

### Yerelleştirilmiş değerleri yükleme ve alma

`Localizations` widget’ı, yerelleştirilmiş değerler koleksiyonlarını
içeren nesneleri yüklemek ve aramak için kullanılır. Uygulamalar bu
nesnelere `Localizations.of(context, type)` ile başvurur. Cihazın yerel
ayarı değişirse, `Localizations` widget’ı yeni yerel ayar için değerleri
otomatik olarak yükler ve ardından onu kullanan widget’ları yeniden
oluşturur. Bu gerçekleşir çünkü `Localizations`, bir `InheritedWidget`
gibi çalışır. Bir derleme (build) işlevi kalıtsal bir widget’a
başvurduğunda, kalıtsal widget üzerinde örtük bir bağımlılık
oluşturulur. Kalıtsal bir widget değiştiğinde (`Localizations`
widget’ının yerel ayarı değiştiğinde), bağımlı bağlamları (contexts)
yeniden oluşturulur.

Yerelleştirilmiş değerler, `Localizations` widget’ının
`LocalizationsDelegates` listesi tarafından yüklenir. Her temsilci
(delegate), yerelleştirilmiş değerler koleksiyonunu kapsayan bir nesne
üreten eşzamansız bir `load()` yöntemi tanımlamalıdır. Tipik olarak bu
nesneler, yerelleştirilmiş değer başına bir yöntem tanımlar.

Büyük bir uygulamada, farklı modüller veya paketler kendi
yerelleştirmeleriyle paketlenebilir. Bu nedenle `Localizations`
widget’ı, `LocalizationsDelegate` başına bir tane olmak üzere bir nesne
tablosunu yönetir. `LocalizationsDelegate`’in `load` yöntemlerinden biri
tarafından üretilen nesneyi almak için bir `BuildContext` ve nesnenin
türünü belirtin.

Örneğin, Materyal Bileşenleri (Material Components) widget’ları için
yerelleştirilmiş dizeler `MaterialLocalizations` sınıfı tarafından
tanımlanır. Bu sınıfın örnekleri, `MaterialApp` sınıfı tarafından
sağlanan bir `LocalizationDelegate` tarafından oluşturulur. Bunlar
`Localizations.of()` ile alınabilir:

``` dart
Localizations.of<MaterialLocalizations>(context, MaterialLocalizations);
```

Bu özel `Localizations.of()` ifadesi sıkça kullanılır, bu nedenle
`MaterialLocalizations` sınıfı uygun bir kısayol sağlar:

``` dart
static MaterialLocalizations of(BuildContext context) {
  return Localizations.of<MaterialLocalizations>(context, MaterialLocalizations);
}

/// MaterialLocalizations tarafından tanımlanan yerelleştirilmiş değerlere referanslar
/// tipik olarak şu şekilde yazılır:

tooltip: MaterialLocalizations.of(context).backButtonTooltip,
```

### Uygulamanın yerelleştirilmiş kaynakları için bir sınıf tanımlama

Uluslararasılaştırılmış bir Flutter uygulamasını bir araya getirmek
genellikle uygulamanın yerelleştirilmiş değerlerini kapsayan sınıfla
başlar. Aşağıdaki örnek, bu tür sınıfların tipik bir örneğidir.

Bu uygulama için `intl_example` tam kaynak kodu.

Bu örnek, `intl` paketi tarafından sağlanan API’lere ve araçlara
dayanmaktadır. “Uygulamanın yerelleştirilmiş kaynakları için alternatif
bir sınıf” bölümü, `intl` paketine bağlı olmayan bir örneği açıklar.

`DemoLocalizations` sınıfı (aşağıdaki kod parçacığında tanımlanmıştır),
uygulamanın desteklediği yerel ayarlara çevrilmiş uygulama dizelerini
(örnek için sadece bir tane) içerir. Bunları aramak için Dart’ın `intl`
paketi tarafından oluşturulan `initializeMessages()` işlevini, yani
`Intl.message()`’ı kullanır.

``` dart
class DemoLocalizations {
  DemoLocalizations(this.localeName);

  static Future<DemoLocalizations> load(Locale locale) {
    final String name =
        locale.countryCode == null || locale.countryCode!.isEmpty
        ? locale.languageCode
        : locale.toString();
    final String localeName = Intl.canonicalizedLocale(name);

    return initializeMessages(localeName).then((_) {
      return DemoLocalizations(localeName);
    });
  }

  static DemoLocalizations of(BuildContext context) {
    return Localizations.of<DemoLocalizations>(context, DemoLocalizations)!;
  }

  final String localeName;

  String get title {
    return Intl.message(
      'Hello World',
      name: 'title',
      desc: 'Title for the Demo application',
      locale: localeName,
    );
  }
}
```

`intl` paketini temel alan bir sınıf, `initializeMessages()` işlevini ve
`Intl.message()` için yerel ayar başına destek deposunu sağlayan
oluşturulmuş bir mesaj kataloğunu içe aktarır. Mesaj kataloğu,
`Intl.message()` çağrılarını içeren sınıflar için kaynak kodunu analiz
eden bir `intl` aracı tarafından üretilir. Bu durumda bu sadece
`DemoLocalizations` sınıfı olacaktır.

### Yeni bir dil desteği ekleme

`GlobalMaterialLocalizations` içinde bulunmayan bir dili desteklemesi
gereken bir uygulamanın fazladan iş yapması gerekir: kelimeler veya
ifadeler için yaklaşık 70 çeviri (“yerelleştirme”) ve yerel ayar için
tarih desenleri ve sembolleri sağlamalıdır.

Norveççe Nynorsk dili için desteğin nasıl ekleneceğine dair bir örnek
için aşağıya bakın.

Yeni bir `GlobalMaterialLocalizations` alt sınıfı, Materyal kitaplığının
bağlı olduğu yerelleştirmeleri tanımlar. `GlobalMaterialLocalizations`
alt sınıfı için fabrika görevi gören yeni bir `LocalizationsDelegate`
alt sınıfı da tanımlanmalıdır.

İşte gerçek Nynorsk çevirileri hariç, tam `add_language` örneğinin
kaynak kodu.

Yerel ayara özgü `GlobalMaterialLocalizations` alt sınıfı
`NnMaterialLocalizations` olarak adlandırılır ve `LocalizationsDelegate`
alt sınıfı `_NnMaterialLocalizationsDelegate`’dir.
`NnMaterialLocalizations.delegate` değeri, temsilcinin bir örneğidir ve
bu yerelleştirmeleri kullanan bir uygulama için gereken tek şeydir.

Temsilci sınıfı, temel tarih ve sayı formatı yerelleştirmelerini içerir.
Diğer tüm yerelleştirmeler, `NnMaterialLocalizations` içinde `String`
değerli özellik alıcıları (getters) tarafından aşağıdaki gibi
tanımlanır:

``` dart
@override
String get moreButtonTooltip => r'More';

@override
String get aboutListTileTitleRaw => r'About $applicationName';

@override
String get alertDialogLabel => r'Alert';
```

Bunlar elbette İngilizce çevirilerdir. İşi tamamlamak için her alıcının
dönüş değerini uygun bir Nynorsk dizesiyle değiştirmeniz gerekir.

Alıcılar, `r` öneki olan “ham” (raw) Dart dizeleri döndürür, örneğin
`r'About $applicationName'`, çünkü bazen dizeler `$` öneki olan
değişkenler içerir. Değişkenler parametreli yerelleştirme yöntemleri
tarafından genişletilir:

``` dart
@override
String get pageRowsInfoTitleRaw => r'$firstRow–$lastRow of $rowCount';

@override
String get pageRowsInfoTitleApproximateRaw =>
    r'$firstRow–$lastRow of about $rowCount';
```

Yerel ayarın tarih desenleri ve sembollerinin de belirtilmesi gerekir,
bunlar kaynak kodunda aşağıdaki gibi tanımlanır:

``` dart
const nnLocaleDatePatterns = {
  'd': 'd.',
  'E': 'ccc',
  'EEEE': 'cccc',
  'LLL': 'LLL',
  // ...
}
const nnDateSymbols = {
  'NAME': 'nn',
  'ERAS': <dynamic>['f.Kr.', 'e.Kr.'],
```

Bu değerlerin, yerel ayarın doğru tarih biçimlendirmesini kullanması
için değiştirilmesi gerekir. Ne yazık ki, `intl` kitaplığı sayı
biçimlendirmesi için aynı esnekliği paylaşmadığından, mevcut bir yerel
ayarın biçimlendirmesi `_NnMaterialLocalizationsDelegate` içinde yedek
olarak kullanılmalıdır:

``` dart
class _NnMaterialLocalizationsDelegate
    extends LocalizationsDelegate<MaterialLocalizations> {
  const _NnMaterialLocalizationsDelegate();

  @override
  bool isSupported(Locale locale) => locale.languageCode == 'nn';

  @override
  Future<MaterialLocalizations> load(Locale locale) async {
    final String localeName = intl.Intl.canonicalizedLocale(locale.toString());

    // Yerel ayarın (bu durumda `nn`) Flutter'ın kullandığı özel tarih sembolleri
    // ve desenleri kurulumuna başlatılması gerekir.
    date_symbol_data_custom.initializeDateFormattingCustom(
      locale: localeName,
      patterns: nnLocaleDatePatterns,
      symbols: intl.DateSymbols.deserializeFromMap(nnDateSymbols),
    );

    return SynchronousFuture<MaterialLocalizations>(
      NnMaterialLocalizations(
        localeName: localeName,
        // `intl` kütüphanesinin NumberFormat sınıfı CLDR verilerinden oluşturulur.
        // Ne yazık ki, bu haritada tanımlanmayan bir yerel ayarı kullanmanın bir yolu yoktur
        // ve bunu aşmanın tek yolu listelenen bir yerel ayarın NumberFormat sembollerini kullanmaktır.
        // Bu yüzden burada 'en_US' için sayı formatlarını kullanıyoruz.
        decimalFormat: intl.NumberFormat('#,##0.###', 'en_US'),
        twoDigitZeroPaddedFormat: intl.NumberFormat('00', 'en_US'),
        // DateFormat burada yukarıdaki `date_symbol_data_custom.initializeDateFormattingCustom`
        // çağrısında sağlanan sembolleri ve desenleri kullanacaktır.
        // Ancak alternatif olarak, yukarıdaki NumberFormat'a benzer şekilde desteklenen
        // bir yerel ayarın DateFormat sembollerini kullanmaktır.
        fullYearFormat: intl.DateFormat('y', localeName),
        compactDateFormat: intl.DateFormat('yMd', localeName),
        shortDateFormat: intl.DateFormat('yMMMd', localeName),
        mediumDateFormat: intl.DateFormat('EEE, MMM d', localeName),
        longDateFormat: intl.DateFormat('EEEE, MMMM d, y', localeName),
        yearMonthFormat: intl.DateFormat('MMMM y', localeName),
        shortMonthDayFormat: intl.DateFormat('MMM d'),
      ),
    );
  }

  @override
  bool shouldReload(_NnMaterialLocalizationsDelegate old) => false;
}
```

Yerelleştirme dizeleri hakkında daha fazla bilgi için
`flutter_localizations` README dosyasına göz atın.

`GlobalMaterialLocalizations` ve `LocalizationsDelegate`’in dile özgü
alt sınıflarını uyguladıktan sonra, dili ve bir temsilci örneğini
uygulamanıza eklemeniz gerekir. Aşağıdaki kod, uygulamanın dilini
Nynorsk olarak ayarlar ve `NnMaterialLocalizations` temsilci örneğini
uygulamanın `localizationsDelegates` listesine ekler:

``` dart
const MaterialApp(
  localizationsDelegates: [
    GlobalWidgetsLocalizations.delegate,
    GlobalMaterialLocalizations.delegate,
    NnMaterialLocalizations.delegate, // Yeni oluşturulan temsilciyi ekleyin
  ],
  supportedLocales: [Locale('en', 'US'), Locale('nn')],
  home: Home(),
),
```

## Alternatif uluslararasılaştırma iş akışları

Bu bölüm, Flutter uygulamanızı uluslararasılaştırmak için farklı
yaklaşımları açıklar.

### Uygulamanın yerelleştirilmiş kaynakları için alternatif bir sınıf

Önceki örnek Dart `intl` paketi açısından tanımlanmıştı. Basitlik adına
veya belki de farklı bir i18n çerçevesiyle entegre olmak için
yerelleştirilmiş değerleri yönetmek üzere kendi yaklaşımınızı
seçebilirsiniz.

`minimal` uygulaması için tam kaynak kodu.

Aşağıdaki örnekte, `DemoLocalizations` sınıfı tüm çevirilerini doğrudan
dil başına Haritalara (Maps) dahil eder:

``` dart
class DemoLocalizations {
  DemoLocalizations(this.locale);

  final Locale locale;

  static DemoLocalizations of(BuildContext context) {
    return Localizations.of<DemoLocalizations>(context, DemoLocalizations)!;
  }

  static const _localizedValues = <String, Map<String, String>>{
    'en': {'title': 'Hello World'},
    'es': {'title': 'Hola Mundo'},
  };

  static List<String> languages() => _localizedValues.keys.toList();

  String get title {
    return _localizedValues[locale.languageCode]!['title']!;
  }
}
```

Minimal uygulamada `DemoLocalizationsDelegate` biraz farklıdır. `load`
yöntemi bir `SynchronousFuture` döndürür çünkü `DemoLocalizations`’ın
bir örneğini üretmek için eşzamansız bir yükleme yapılması gerekmez.

``` dart
class DemoLocalizationsDelegate
    extends LocalizationsDelegate<DemoLocalizations> {
  const DemoLocalizationsDelegate();

  @override
  bool isSupported(Locale locale) =>
      DemoLocalizations.languages().contains(locale.languageCode);

  @override
  Future<DemoLocalizations> load(Locale locale) {
    // DemoLocalizations örneği üretmek için eşzamansız bir "load" işlemi gerekmediğinden
    // burada bir SynchronousFuture döndürülüyor.
    return SynchronousFuture<DemoLocalizations>(DemoLocalizations(locale));
  }

  @override
  bool shouldReload(DemoLocalizationsDelegate old) => false;
}
```

### Dart intl araçlarını kullanma

Dart `intl` paketini kullanarak bir API oluşturmadan önce `intl`
paketinin belgelerini inceleyin. Aşağıdaki liste, `intl` paketine bağlı
bir uygulamayı yerelleştirme sürecini özetlemektedir:

Demo uygulaması, uygulama tarafından kullanılan tüm yerelleştirilebilir
dizeleri tanımlayan `l10n/messages_all.dart` adlı oluşturulmuş bir
kaynak dosyasına bağlıdır.

`l10n/messages_all.dart` dosyasını yeniden oluşturmak iki adım
gerektirir.

Uygulamanın kök dizini geçerli dizin olacak şekilde, `lib/main.dart`
dosyasından `l10n/intl_messages.arb` dosyasını oluşturun:

``` terminal
dart run intl_translation:extract_to_arb --output-dir=lib/l10n lib/main.dart
```

`intl_messages.arb` dosyası, `main.dart` içinde tanımlanan her
`Intl.message()` işlevi için bir giriş içeren JSON formatında bir
haritadır. Bu dosya, İngilizce ve İspanyolca çeviriler olan
`intl_en.arb` ve `intl_es.arb` için bir şablon görevi görür. Bu
çeviriler siz, geliştirici tarafından oluşturulur.

Uygulamanın kök dizini geçerli dizin olacak şekilde, her
`intl_<locale>.arb` dosyası için `intl_messages_<locale>.dart` dosyasını
ve tüm mesaj dosyalarını içe aktaran `intl_messages_all.dart` dosyasını
oluşturun:

``` terminal
dart run intl_translation:generate_from_arb \
    --output-dir=lib/l10n --no-use-deferred-loading \
    lib/main.dart lib/l10n/intl_*.arb
```

Windows dosya adı joker karakterlerini (wildcarding) desteklemez. Bunun
yerine, `intl_translation:extract_to_arb` komutuyla oluşturulan .arb
dosyalarını listeleyin.

``` terminal
dart run intl_translation:generate_from_arb \
    --output-dir=lib/l10n --no-use-deferred-loading \
    lib/main.dart \
    lib/l10n/intl_en.arb lib/l10n/intl_fr.arb lib/l10n/intl_messages.arb
```

`DemoLocalizations` sınıfı, yerelleştirilmiş mesajları yüklemek için
(`intl_messages_all.dart` içinde tanımlanan) oluşturulmuş
`initializeMessages()` işlevini ve bunları aramak için
`Intl.message()`’ı kullanır.

### Daha fazla bilgi

Kod okuyarak en iyi şekilde öğreniyorsanız, aşağıdaki örneklere göz
atın.

- [minimal](https://github.com/flutter/website/tree/main/examples/internationalization/minimal):
  `minimal` örneği mümkün olduğunca basit olacak şekilde tasarlanmıştır.
- [intl_example](https://github.com/flutter/website/tree/main/examples/internationalization/intl_example):
  `intl` paketi tarafından sağlanan API’leri ve araçları kullanır.

Dart’ın `intl` paketi sizin için yeniyse, [Dart intl araçlarını
kullanma](https://docs.flutter.dev/ui/internationalization#dart-tools)
bölümüne göz atın.

## 📄 Lisans Bilgisi

Bu doküman, **Flutter resmi dokümantasyonundan** türetilmiş Türkçe ders
notudur.

**Orijinal kaynak:**  
https://docs.flutter.dev/ui/internationalization

**Türkçe çeviri ve düzenleme:**  
[Doç. Dr. Hakan Temiz](mailto:htemiz@artvin.edu.tr)

------------------------------------------------------------------------

### Lisans Kapsamı

Bu dokümandaki içerikler aşağıdaki açık lisanslar kapsamında
sunulmaktadır:

**Metin içerikleri (anlatım ve açıklamalar):**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** Creative Commons Attribution 4.0 International (CC BY 4.0)  
https://creativecommons.org/licenses/by/4.0/

Bu lisans kapsamında: - İçerik kopyalanabilir, dağıtılabilir ve
uyarlanabilir  
- Ticari kullanım serbesttir  
- Kaynak belirtilmesi zorunludur

**Kod örnekleri:**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** BSD 3-Clause License  
https://opensource.org/licenses/BSD-3-Clause

Bu lisans kapsamında: - Kodlar kopyalanabilir, değiştirilebilir ve
dağıtılabilir  
- Ticari kullanım serbesttir  
- Lisans bildiriminin korunması gerekir